In [1]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
!pip install optuna
import optuna
from torch.utils.data import random_split

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 12.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Load Data

In [4]:
image_transforms=transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2,contrast=0.2),
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
import zipfile
import os

zip_path="/content/drive/MyDrive/Colab Notebooks/dataset.zip"
extract_path="./dataset_extracted"

# Create the extraction directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip the dataset
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Update dataset_path to point to the nested 'dataset' folder
dataset_path = os.path.join(extract_path, 'dataset')

dataset=datasets.ImageFolder(root=dataset_path,transform=image_transforms)
len(dataset)

2300

In [6]:
class_names=dataset.classes
class_names

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [7]:
num_classes=len(dataset.classes)
num_classes

6

In [8]:
train_size=int(0.75*len(dataset))
val_size=int(0.25*len(dataset))

train_size,val_size

(1725, 575)

In [9]:
train_dataset,val_dataset=random_split(dataset,[train_size,val_size])

In [10]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=False)

## Model Training and Hyperparameter Tuning

In [11]:
#Load the pretrained ResNet model

class CarClassifierResNet(nn.Module):
    def __init__(self,num_classes,dropout_rate=0.5):
        super().__init__()
        self.model=models.resnet50(weights="DEFAULT")

        #Freeze all layers except final fully connected layer

        for param in self.model.parameters():
            param.requires_grad=False

        #Unfreeze layer4 and fc layer
        for param in self.model.layer4.parameters():
            param.requires_grad=True

        #Replace the final fully connected layer
        self.model.fc=nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.model.fc.in_features,num_classes)
        )

    def forward(self,x):
        x=self.model(x)
        return x

In [12]:
#Define the objective function of optuna

def objective(trial):
    #suggest values for parameters
    lr=trial.suggest_float('lr',1e-5,1e-2,log=True)
    dropout_rate=trial.suggest_float('dropout_rate',0.2,0.7)

    #Load the model
    model=CarClassifierResNet(num_classes=num_classes,dropout_rate=dropout_rate).to(device)

    #Define the loss function and optimizer
    criterion=nn.CrossEntropyLoss()
    optimizer=optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=lr)

    #Training loop (using fewer epochs for faster hyperparameter tuning)
    epochs=3
    start=time.time()
    for epoch in range(epochs):
        model.train()
        running_loss=0.0
        for batch_num,(images,labels) in enumerate(train_loader):
            images,labels=images.to(device),labels.to(device)

            optimizer.zero_grad()
            outputs=model(images)
            loss=criterion(outputs,labels)

            loss.backward()
            optimizer.step()

            running_loss+=loss.item()*images.size(0)
        epoch_loss=running_loss/len(train_loader.dataset)

        #Validation loop
        model.eval()
        correct=0
        total=0
        with torch.no_grad():
            for images,labels in val_loader:
                images,labels=images.to(device),labels.to(device)

                outputs=model(images)
                _,predicted=torch.max(outputs.data,1)
                total+=labels.size(0)
                correct+=(predicted==labels).sum().item()
        accuracy=(correct*100)/total

        #Report intermediate result to optuna
        trial.report(accuracy,epoch)

        #handle pruning is applicable
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time: {end - start} seconds")

    return accuracy


In [13]:
study=optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=20)

[I 2026-05-22 07:34:23,207] A new study created in memory with name: no-name-a0cf9465-6b67-4490-b08b-2f4e1d391f13


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 91.8MB/s]
[I 2026-05-22 07:38:09,721] Trial 0 finished with value: 76.52173913043478 and parameters: {'lr': 0.0025801504687573335, 'dropout_rate': 0.33652178134083055}. Best is trial 0 with value: 76.52173913043478.


Execution time: 223.96451616287231 seconds


[I 2026-05-22 07:41:53,651] Trial 1 finished with value: 69.21739130434783 and parameters: {'lr': 0.0056272525951537086, 'dropout_rate': 0.2931024339382027}. Best is trial 0 with value: 76.52173913043478.


...

[I 2026-05-22 08:18:25,112] Trial 14 pruned. 
[I 2026-05-22 08:22:07,467] Trial 15 finished with value: 78.78260869565217 and parameters: {'lr': 0.0007720523959355113, 'dropout_rate': 0.6944856623217524}. Best is trial 8 with value: 80.0.


Execution time: 221.92450952529907 seconds


[I 2026-05-22 08:23:22,194] Trial 16 pruned. 
[I 2026-05-22 08:24:38,056] Trial 17 pruned. 
[I 2026-05-22 08:28:24,222] Trial 18 finished with value: 79.30434782608695 and parameters: {'lr': 0.0007322624150170063, 'dropout_rate': 0.437259735683705}. Best is trial 8 with value: 80.0.


Execution time: 225.70386743545532 seconds


[I 2026-05-22 08:29:40,818] Trial 19 pruned. 


In [14]:
study.best_params

{'lr': 0.0005996423257915503, 'dropout_rate': 0.49867364936572045}